# AED-XAI: Local Setup & Data Preparation
> Run this notebook once to set up your environment, download COCO,
> and verify the pipeline is working correctly.


In [ ]:
import subprocess, sys, torch

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found. VLM inference will be very slow on CPU.")


## 1. Clone the Repository
Skip this cell if you already have the repo cloned locally.


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/sigango/aedxai.git"
HOME_REPO_DIR = Path.home() / "aedxai"
cwd = Path.cwd().resolve()

repo_dir = None
for candidate in (cwd, cwd.parent):
    if (candidate / "requirements.txt").exists() and (candidate / "src").exists():
        repo_dir = candidate
        break

if repo_dir is None:
    repo_dir = HOME_REPO_DIR
    if not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
        print(f"Cloned to {repo_dir}")
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)
        print(f"Updated existing repo at {repo_dir}")
else:
    print(f"Using existing repo at {repo_dir}")

# Change working directory to repo root for all subsequent cells
os.chdir(repo_dir)
print(f"Working directory: {os.getcwd()}")


## 2. Install Dependencies

We recommend a fresh conda environment:
```bash
conda create -n aedxai python=3.10 -y
conda activate aedxai
```
Then run the cell below inside that environment.


In [ ]:
# Install all requirements
subprocess.run([sys.executable, "-m", "pip", "install", "--no-build-isolation", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

# Install YOLOX from source (not on PyPI)
result = subprocess.run(
    [sys.executable, "-m", "pip", "show", "yolox"],
    capture_output=True, text=True
)
if "Name: yolox" not in result.stdout:
    print("Installing YOLOX from source...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--no-build-isolation",
        "git+https://github.com/Megvii-BaseDetection/YOLOX.git"
    ], check=True)
    print("YOLOX installed.")
else:
    print("YOLOX already installed.")


## 3. Download COCO Dataset

AED-XAI uses COCO val2017 for evaluation and COCO train2017 for
selector training.

| Split | Size | Use |
|-------|------|-----|
| val2017 images | ~1 GB | Pipeline evaluation, quick tests |
| train2017 images | ~18 GB | Selector oracle label generation |
| annotations (all) | ~241 MB | Bounding box ground truth |

Set `DOWNLOAD_TRAIN = False` to skip the 18GB training set if you only
want to run inference/evaluation first.


In [ ]:
import os
import zipfile
import urllib.request
from pathlib import Path
from tqdm.auto import tqdm

# ── Configuration ──────────────────────────────────────────────────
DATA_DIR = Path("data/coco")
DOWNLOAD_TRAIN = False   # Set True to download train2017 (~18GB)
# ───────────────────────────────────────────────────────────────────

DATA_DIR.mkdir(parents=True, exist_ok=True)

URLS = {
    "val2017.zip":         "http://images.cocodataset.org/zips/val2017.zip",
    "annotations_trainval2017.zip": "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
}
if DOWNLOAD_TRAIN:
    URLS["train2017.zip"] = "http://images.cocodataset.org/zips/train2017.zip"

class _ProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

def download_and_extract(url: str, dest_dir: Path) -> None:
    filename = url.split("/")[-1]
    zip_path = dest_dir / filename
    extract_name = filename.replace(".zip", "")
    extract_path = dest_dir / extract_name

    # Skip if already extracted
    if extract_path.exists():
        print(f"  Already extracted: {extract_path}")
        return

    # Download
    if not zip_path.exists():
        print(f"  Downloading {filename} ...")
        with _ProgressBar(unit="B", unit_scale=True, miniters=1, desc=filename) as pb:
            urllib.request.urlretrieve(url, zip_path, reporthook=pb.update_to)
        print(f"  Downloaded {zip_path}")
    else:
        print(f"  Already downloaded: {zip_path}")

    # Extract
    print(f"  Extracting {filename} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest_dir)
    print(f"  Extracted to {extract_path}")

for url in URLS.values():
    download_and_extract(url, DATA_DIR)

print("\nCOCO directory structure:")
for p in sorted(DATA_DIR.iterdir()):
    n = sum(1 for _ in p.rglob("*") if _.is_file()) if p.is_dir() else 1
    print(f"  {p.name}/  ({n} files)")


## 4. Verify COCO Annotations


In [ ]:
from pycocotools.coco import COCO
import numpy as np

ann_file = DATA_DIR / "annotations" / "instances_val2017.json"
coco_val = COCO(str(ann_file))

img_ids = coco_val.getImgIds()
print(f"val2017 images : {len(img_ids):,}")
print(f"categories     : {len(coco_val.getCatIds())}")

# Sample one image and show its annotations
sample_id = img_ids[0]
info = coco_val.loadImgs(sample_id)[0]
ann_ids = coco_val.getAnnIds(imgIds=sample_id)
anns = coco_val.loadAnns(ann_ids)
print(f"\nSample image   : {info['file_name']}")
print(f"Annotations    : {len(anns)} objects")
for a in anns[:3]:
    cat = coco_val.loadCats(a['category_id'])[0]['name']
    print(f"  {cat}: bbox={[round(x,1) for x in a['bbox']]}")


## 5. Verify Pipeline Imports


In [ ]:
# These imports confirm the package is installed correctly
from src.detector import DetectorWrapper
from src.vlm_judge import VLMJudge
from src.xai_selector import XAISelector
from src.evaluator import AutoEvaluator
from src.feedback_loop import FeedbackLoop
from src.pipeline import AEDXAIPipeline
from src.threshold import AdaptiveThreshold
from src.xai_methods import get_explainer
from src.xai_methods.gradcam import GradCAMExplainer
from src.xai_methods.gcame import GCAMEExplainer
from src.xai_methods.dclose import DCLOSEExplainer
from src.xai_methods.lime_det import LIMEExplainer

print("All imports OK")

# Verify composite weights are 1/3 each
e = AutoEvaluator()
w = e.composite_weights
print(f"\nComposite weights: {w}")
assert abs(sum(w.values()) - 1.0) < 0.01, "Weights must sum to 1.0"
assert all(abs(v - 1/3) < 0.01 for v in w.values()), "All weights should be ~1/3"
print("Composite weights verified: equal 1/3 each ✓")


## 6. Quick Detector Smoke Test

Runs YOLOX-S on one COCO val2017 image to verify the detector loads
and produces detections. Does NOT load the VLM.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from src.utils import load_image

# Pick the first val2017 image
img_info = coco_val.loadImgs(img_ids[0])[0]
img_path = str(DATA_DIR / "val2017" / img_info["file_name"])

print(f"Loading detector (YOLOX-S)...")
detector = DetectorWrapper(model_name="yolox-s", config_path="config/detector_config.yaml")
detector.load_model()
print("Detector loaded.")

image = load_image(img_path)
detections = detector.detect(image)

print(f"Image: {img_info['file_name']} ({image.shape[1]}x{image.shape[0]})")
print(f"Detections: {len(detections)}")
for d in detections[:5]:
    print(f"  [{d.class_name}] conf={d.confidence:.2f} bbox={[round(x) for x in d.bbox]}")

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
ax.imshow(image)
ax.set_title(f"YOLOX-S detections: {len(detections)} objects")
ax.axis("off")
for d in detections:
    x1, y1, x2, y2 = d.bbox
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=2, edgecolor="lime", facecolor="none"
    )
    ax.add_patch(rect)
    ax.text(x1, y1 - 4, f"{d.class_name} {d.confidence:.2f}",
            color="lime", fontsize=8, fontweight="bold")
plt.tight_layout()
plt.show()

# Clean up GPU memory before next steps
detector.unload_model()
import gc; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Detector unloaded.")


## 7. VRAM Budget Check

AED-XAI loads multiple models. This cell estimates VRAM usage for RTX 3090 (24GB).


In [ ]:
print("=== VRAM Budget Estimate (RTX 3090, 24GB) ===")
budget = {
    "YOLOX-S (FP32)":         "~0.4 GB",
    "Qwen2.5-VL-7B (INT4)":   "~4.5 GB",
    "GradCAM activations":     "~0.5 GB",
    "G-CAME activations":      "~0.5 GB",
    "D-CLOSE perturbations":   "~1.0 GB (batch of 10)",
    "LIME perturbations":      "~0.5 GB (batch of 10)",
    "PyTorch overhead":        "~1.0 GB",
    "─────────────────────":  "──────",
    "TOTAL (worst case)":      "~8.4 GB  (well within 24GB)",
}
for k, v in budget.items():
    print(f"  {k:<30} {v}")

if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nYour GPU: {torch.cuda.get_device_name(0)} ({total_gb:.1f} GB)")
    if total_gb >= 20:
        print("✓ Sufficient VRAM for full pipeline.")
    elif total_gb >= 12:
        print("⚠ Marginal. Disable VLM or use INT4 quantization.")
    else:
        print("✗ Insufficient VRAM for full pipeline. CPU fallback only.")


## 8. Next Steps

After this notebook completes successfully:

1. **Run the full experiment orchestration notebook** to produce the selector ablation, 3000-image evaluation, detector comparison, cross-domain CSVs, and figure inputs:
   ```bash
   jupyter lab notebooks/04_run_all_experiments.ipynb
   ```

2. **Or run selector training from the CLI** for both detector families before the main evaluation. The oracle CSV is intentionally larger than the production selector size so the ablation can sweep `50, 100, 200, 500, 1000`, while the main AED-XAI run uses a 500-row selector checkpoint:
   ```bash
   for detector in yolox-s fasterrcnn_resnet50_fpn_v2; do
       python scripts/train_selector.py \
           --detector-model "$detector" \
           --max-images 1000 \
           --target-detections 2000 \
           --checkpoint-every 100 \
           --oracle-mode \
           --resume
   done
   ```

3. **Run the full AED-XAI evaluation** on COCO val2017 using 3000 images:
   ```bash
   python scripts/run_experiments.py \
       --images-dir data/coco/val2017 \
       --num-images 3000 \
       --output results/aedxai \
       --detector-model yolox-s
   ```

4. **Run existing tests** to verify nothing is broken:
   ```bash
   pytest tests/ -v --ignore=tests/test_pipeline.py -m "not slow"
   ```
